In [ ]:
from IPython.display import Image

image_url = "https://cdn.prod.website-files.com/65c81e88c254bb0f97633a71/69cbb087f72d73b2dc0182eb_LangChain-Stack---split---V3.png"

Image(url=image_url, width=1000, height=1000)

# Section 1: Dependencies Installation

Explanation: In this section, we install the necessary libraries for building our chatbot, including LangChain, Streamlit, Pyngrok, and the Ollama package.

In [1]:
!pip install -U langchain langchain-community langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 591.1/591.1 kB 14.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 59.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 41.6 MB/s  0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-protocol
    Found existing installation: langchain-protocol 0.0.20
    Uninstalling langchain-protocol-0.0.20:
      Successfully uninstalled langchain-protocol-0.0.20
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.7.4
    Uninstalling langchain-core-1.7.4:
      Successfully uninstalled langchain-core-1.7.4
  Attempting uninstall: langchain
    Found existing installation: langchain 1.4.8
    Uninstalling langchain-1.4.8:
      Successfully uninstalled langchain-1.4.8
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10/10 [langchain

Explanation: Installs Streamlit to build the interactive web user interface.

In [2]:
!pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 20.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 50.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [streamlit]


Explanation: Installs the `deepagents` package containing helper frameworks and specific API integrations like Google Gemini.

In [3]:
!pip install -U deepagents

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 21.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 55.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 55.7 MB/s  0:00:00
  Attempting uninstall: langsmith
    Found existing installation: langsmith 0.14.17
    Uninstalling langsmith-0.14.17:
      Successfully uninstalled langsmith-0.14.17
  Attempting uninstall: google-auth
    Found existing installation: google-auth 2.49.0
    Uninstalling google-auth-2.49.0:
      Successfully uninstalled google-auth-2.49.0
  Attempting uninstall: google-genai
    Found existing installation: google-genai 2.12.1
    Uninstalling google-genai-2.12.1:
      Successfully uninstalled google-genai-2.12.1
  Attempting uninstall: langgraph-prebuilt
    Found existing installation: langgraph-prebuilt 1.1.1
    Uninstalling langgraph-prebuilt-1.1.1:
      Successfully uninstalled langgraph-prebuilt-1.1.1
  Attempting uninstall: langgraph
    Found existing i

Explanation: Installs `pyngrok` to securely expose our locally running Streamlit app to a public URL.

In [4]:
!pip install -q pyngrok

In [ ]:
from pyngrok import ngrok

NGROK_AUTHTOKEN = "<NGROK-AUTHTOKEN>"

ngrok.set_auth_token(NGROK_AUTHTOKEN)

# Section 4: Streamlit Applications

Explanation: In this section, we write the Streamlit application code and launch the background server connected to Ngrok.

In [18]:
%%writefile app.py
import streamlit as st
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Configure Environment Variables
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_API_KEY"] = "<LANGSMITH-API-KEY>"
os.environ["LANGCHAIN_PROJECT"] = "langchain-first-chatbot"
os.environ["GOOGLE_API_KEY"] = "<GOOGLE-API-KEY>"

# Streamlit Page UI
st.title("LangChain demo with Gemini API")
input_text = st.text_input("Search the topic you want")

# Build the Chain
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Please respond to the user queries."),
    ("user", "Question: {question}")
])

# Model - Using Gemini 3.5 Flash
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash")
output_parser = StrOutputParser()
chain = prompt | llm | output_parser

if input_text:
    response = chain.invoke({'question': input_text})
    st.write(response)

Overwriting app.py


Explanation: Creates `app.py` which defines a Streamlit UI connected to the local Llama2 model via LangChain.

In [19]:
import subprocess
import time
from pyngrok import ngrok

# Terminate existing Streamlit processes & active tunnels
subprocess.run("pkill streamlit", shell=True)
ngrok.kill()

# Start Streamlit background server
subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.address", "0.0.0.0"])
time.sleep(3)

# Connect Ngrok tunnel to Streamlit port 8501
public_url = ngrok.connect(8501, "http")
print(f"Streamlit App is live at: {public_url}")

Streamlit App is live at: NgrokTunnel: "https://surfacing-parakeet-immerse.ngrok-free.dev" -> "http://localhost:8501"


# Section 2: Local Ollama Setup

Explanation: This section installs the Ollama service, starts its daemon process in the background, and downloads the open-source Llama2 model.

In [20]:
!pip install -U ollama

In [25]:
!sudo apt-get update && sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

Hit:1 http://archive.ubuntu.com/ubuntu noble InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Get:4 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:5 http://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Get:6 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Get:8 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ Packages [82.5 kB]
Get:9 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:10 http://archive.ubuntu.com/ubuntu noble-updates/multiverse amd64 Packages [61.7 kB]
Get:11 http://archive.ubuntu.com/ubuntu noble-updates/restricted amd64 Packages [2,403 kB]
Get:12 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 Packages [1,843 kB]
Get:13 http://archive.ubuntu.com/ubuntu noble-updates/universe amd64 Packages [2,18

Explanation: Launches the background Ollama server process so it can handle API requests on port 11434.

In [27]:
import subprocess
import time

print("Starting Ollama server...")
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

Starting Ollama server...


Explanation: Downloads the Llama2 model weights to Colab so we can perform offline local inference.

In [28]:
print("Downloading llama2 model (this may take a few minutes)...")
!ollama pull llama2
print("Done! Ollama is now ready to serve queries.")


Done! Ollama is now ready to serve queries.


# Section 3: Ngrok Authentication

Explanation: Here we configure the Ngrok tunnel credentials using your personal authtoken to allow external web traffic.

In [29]:
%%writefile localollama.py
import streamlit as st
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.llms import Ollama

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_API_KEY"] = "<LANGSMITH-API-KEY>"
os.environ["LANGCHAIN_PROJECT"] = "langchain-first-chatbot"
os.environ["GOOGLE_API_KEY"] = "<GOOGLE-API-KEY>"

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are helpful assistant. please response to user queries"),
        ("user", "Question: {question}")
    ]
)

st.title("LangChain Chatbot using OLAMA")
input_text = st.text_input("Search the topic you want...")

llm = Ollama(model = "llama2")
output_parser = StrOutputParser()
chain = prompt|llm|output_parser

if input_text:
  st.write(chain.invoke({"question": input_text}))

Overwriting localollama.py


Explanation: Shuts down any conflicting ports, runs the Streamlit server on port 8501, and opens the secure Ngrok tunnel.

In [30]:
import subprocess
import time
from pyngrok import ngrok

# Terminate existing Streamlit processes & active tunnels
subprocess.run("pkill streamlit", shell=True)
ngrok.kill()

# Start Streamlit background server
subprocess.Popen(["streamlit", "run", "localollama.py", "--server.port", "8501", "--server.address", "0.0.0.0"])
time.sleep(3)

# Connect Ngrok tunnel to Streamlit port 8501
public_url = ngrok.connect(8501, "http")
print(f"Streamlit App is live at: {public_url}")

Streamlit App is live at: NgrokTunnel: "https://surfacing-parakeet-immerse.ngrok-free.dev" -> "http://localhost:8501"
